# Library

In [1]:
import tensorflow as tf
import math
from tqdm import tqdm
import numpy as np
import time
import cv2
import pickle
import matplotlib.pyplot as plt
from PIL import Image
import scipy as scipy
from collections import Counter
from Erosion import crop_rice_grains
import uuid
import gc
import os
import pandas as pd


from tensorflow.keras.models import Model,load_model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input,Dense, Conv2D,GlobalAveragePooling2D,Dropout,Flatten,BatchNormalization,Concatenate,InputLayer,AveragePooling2D
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import AdamW

2025-03-25 13:16:42.319386: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742883402.336336 1778698 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742883402.341724 1778698 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-25 13:16:42.360993: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
path_to_MP_India = "/media/new_volumn/MP_India"

# Search for all subfolders (classes) in "PNG_Paddy07-2024/"
def find_subfolders(base_dir):
        subfolders = []
        for root, dirs, files in os.walk(base_dir): 
            for dir in dirs:
                full_path = os.path.join(root, dir) # full path to the subfolder
                if os.listdir(full_path): # Check if the subfolder is not empty
                        subfolders.append(full_path)
        return subfolders

all_subfolders = find_subfolders(path_to_MP_India)
all_subfolders = sorted(all_subfolders)

In [3]:
len(all_subfolders), all_subfolders[:10]

(83,
 ['/media/new_volumn/MP_India/000001',
  '/media/new_volumn/MP_India/0000010',
  '/media/new_volumn/MP_India/0000011',
  '/media/new_volumn/MP_India/0000012',
  '/media/new_volumn/MP_India/0000013',
  '/media/new_volumn/MP_India/0000014',
  '/media/new_volumn/MP_India/0000015',
  '/media/new_volumn/MP_India/0000016',
  '/media/new_volumn/MP_India/0000017',
  '/media/new_volumn/MP_India/0000018'])

# Initialize Model

In [4]:
gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    # Restrict TensorFlow to only use the first GPU
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, False)
            tf.config.experimental.set_virtual_device_configuration(
                gpu,
                [
                    tf.config.experimental.VirtualDeviceConfiguration(
                        memory_limit=18288  # set your limit
                    )
                ],
            )
        tf.config.experimental.set_visible_devices(gpus[0], "GPU")
        logical_gpus = tf.config.experimental.list_logical_devices("GPU")
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPU")
    except RuntimeError as e:
        # Visible devices must be set before GPUs have been initialized
        print(e)

1 Physical GPUs, 1 Logical GPU


I0000 00:00:1742883411.876618 1778698 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18288 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


In [5]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

config = base_model.get_config()
new_model = tf.keras.models.Model.from_config(config) # <- new_model architecture

x_model = new_model.output
# x_model = data_augmentation(x_model)  
x_model = GlobalAveragePooling2D()(x_model)
x_model = BatchNormalization()(x_model)
x_model = Dropout(0.3)(x_model)
x_model = Dense(128,activation='relu')(x_model)
x_model = BatchNormalization()(x_model)
x_model = Dense(128,activation='relu')(x_model)
x_model = Dropout(0.3)(x_model)
x_model = Dense(64,activation='relu')(x_model)
x_model = Dropout(0.3)(x_model)

predictions = Dense(5, activation='softmax')(x_model)
model = Model(inputs=new_model.input, outputs=predictions)

model.load_weights('/media/new_volumn/MP_DEFECT/models/defect/modelsB0V4.keras')

In [6]:
# A set to keep track of generated UUIDs for uniqueness check
generated_uuids = set()

def evaluate(X_test, npz_prefix,subfolder_name, npz_filename):
    os.makedirs(f"/media/new_volumn/MP_India_Cleaned1/{subfolder_name}", exist_ok=True)
    
    list_normal_rice = []

    data_dict = [
                    'broke',
                    'dual',
                    'half',
                    'normal',
                    'over'
                ]
    # Perform batch prediction
    Y_pred = model.predict(X_test, verbose=0) 
    y_pred = np.argmax(Y_pred, axis=1)  # Get class indices for the whole batch

    for i, (image, pred_class) in enumerate(zip(X_test, y_pred)):
        class_label = data_dict[pred_class]  # Get class name from index

        # # Generate unique ID
        # unique_id = uuid.uuid4().hex
        # while unique_id in generated_uuids:
        #     unique_id = uuid.uuid4().hex
        # generated_uuids.add(unique_id)

        # Convert image to NumPy and save
        image_pil = Image.fromarray((image * 255).astype(np.uint8)) 

        # Append normal kernels to the list
        if class_label == "normal":
            list_normal_rice.append(image_pil)

    list_normal_rice = np.array(list_normal_rice)
    np.savez_compressed(f"/media/new_volumn/MP_India_Cleaned1/{subfolder_name}/{npz_filename}", kernel_pics=list_normal_rice)

    return list_normal_rice

In [7]:
all_subfolders[:2]

['/media/new_volumn/MP_India/000001', '/media/new_volumn/MP_India/0000010']

In [8]:
data = []
err_count = 0
for subfolder in tqdm(all_subfolders, desc="looping through subfolders"):
    total_images = len(os.listdir(subfolder))
    total_kernels = 0
    final_kernels = 0
    # Loop through each image in the subfolder
    for npz in tqdm(os.listdir(subfolder), desc="looping through images", leave=False):
        sample_images = np.load(f"{subfolder}/{npz}")['kernel_pics'].astype('uint8')
        total_kernels += sample_images.shape[0]
        npz_prefix = npz.split("/")[-1][:-4] # obtain the prefix before .npz
        X_test, npz_prefix, err = crop_rice_grains(sample_images, npz_prefix) # apply erosion and dilation, and normalization
        if err: # if error occurs, skip the current npz filee
            err_count += 1
            print(f"Error in {npz}, Skipping...")
            total_images -= 1
            continue
        npz_filename = npz.split("/")[-1] # obtain original npz filename
        subfolder_name = subfolder.split("/")[-1] # obtain the subfolder name
        normal_kernels = evaluate(X_test, npz_prefix, subfolder_name,npz_filename)
        final_kernels += normal_kernels.shape[0]
    # Append the collected data
    data.append({
        "Subfolder": subfolder,
        "Total Images": total_images,
        "Total Kernels": total_kernels,
        "Final Kernels": final_kernels
    })


del model
gc.collect()

looping through subfolders:   0%|                                                                                   | 0/83 [00:00<?, ?it/s]
I0000 00:00:1742883434.637884 1779026 service.cc:148] XLA service 0x7fb548002300 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742883434.637925 1779026 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-03-25 13:17:14.718203: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1742883435.059601 1779026 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-03-25 13:17:15.209150: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may sl

Unexpected image shape: (0,), the source npz file is IMG_20240223_0001
Error in IMG_20240223_0001.npz, Skipping...



looping through subfolders: 100%|████████████████████████████████████████████████████████████████████████| 83/83 [1:06:47<00:00, 48.28s/it]


16264

In [11]:
df = pd.DataFrame(data)
df.to_csv('metadata.csv', index=False)  

In [3]:
import pandas as pd
df = pd.read_csv('metadata.csv')

In [7]:
df.sort_values('Subfolder', inplace=True)

In [ ]:
df

In [ ]:
# def evaluate(X_test, data_dict, subfolder_name, npz_filename):
#     os.makedirs(f"/media/new_volumn/MP_India_Cleaned/{subfolder_name}", exist_ok=True)
    
#     Y_pred = model.predict(X_test, verbose=0)  
#     y_pred = np.argmax(Y_pred, axis=1)  

#     # Filter "normal" rice kernels efficiently using NumPy indexing
#     normal_indices = np.where(y_pred == data_dict.index("normal"))[0]
#     list_normal_rice = (X_test[normal_indices] * 255).astype(np.uint8)

#     # Save as .npz
#     np.savez_compressed(
#         f"/media/new_volumn/MP_India_Cleaned/{subfolder_name}/{npz_filename}.npz",
#         kernel_pics=list_normal_rice
#     )

#     return len(list_normal_rice)  # Return the count instead of full array

# def process_npz(npz_info):
#     subfolder, npz = npz_info
#     try:
#         sample_images = np.load(f"{subfolder}/{npz}")['kernel_pics'].astype('uint8')
#         X_test, npz_prefix, err = crop_rice_grains(sample_images, npz.split("/")[-1][:-4]) 

#         if err:
#             return {"Subfolder": subfolder, "Skipped": True}

#         subfolder_name = subfolder.split("/")[-1]  
#         final_kernels = evaluate(X_test, ['broke', 'dual', 'half', 'normal', 'over'], subfolder_name, npz)
        
#         return {
#             "Subfolder": subfolder,
#             "Skipped": False,
#             "Total Kernels": sample_images.shape[0],
#             "Final Kernels": final_kernels
#         }
#     except Exception as e:
#         return {"Subfolder": subfolder, "Skipped": True, "Error": str(e)}

# all_npz_files = [(subfolder, npz) for subfolder in all_subfolders for npz in os.listdir(subfolder)]
# with Pool(processes=4) as pool:  # Adjust the number of processes as needed
#     results = list(tqdm(pool.imap(process_npz, all_npz_files), total=len(all_npz_files), desc="Processing files"))

# # Collect results
# data = [res for res in results if not res.get("Skipped")]